# 第17回　総合課題 — 手書き数字認識
***
> **前提**: 第16回で学習・保存した `best_mnist_model.pth` を使い，自作の手書き数字で推論します。
>
> **実行環境**: 手書き推論 UI は **Google Colab** 上での実行を推奨します（`google.colab.output` を使用）。Colab では第14〜16回を順に実行してから本課題に取り組んでください。ローカル Jupyter では問題1と問題3（考察）のみ実施可能です。

> ⚠️ **この課題で身につけること：コーディングではなく「AI（ニューラルネット）の中身の理解」です。**
>
> コードは AI に書かせても構いません。重要なのは「**なぜその処理を選ぶのか**」「**パラメータや特徴量を変えると結果がどう変わるのか**」を理解し、提出物で示すことです。本回は **考察中心**です。各問には学習目標を示すタグが付いています。
>
> | タグ | 意味 | あなたがすること |
> |---|---|---|
> | **【骨格】** | 動くコードは与えられている | 設計上の決定点（数値・選択肢・特徴量）だけを変更する |
> | **【選択】** | 適切な手法を選ぶ問題 | 複数候補から選び、**理由**を解答用コードセルに書く |
> | **【実験】** | 試行錯誤の記録 | パラメータ等を変えて結果を表に記録し、**考察**する |
> | **【説明】** | 理解の証跡 | 与えられたコードの各行に `# 説明:` で意味を書く |
> | **【考察】** | 理解の証跡 | なぜそうなるのかを自分の言葉で説明する |
>
> コードは原則として完成形ですが、**核心となる最低限の数行は `# ★あなたが書く★` として空欄**にしてあります。要となる処理は自分で書けることも確認します（ボイラープレートは提供済み）。
>
> 各問の **✍️ 解答用コードセル**（`# (1-a)` 形式の変数・文字列）に、選んだ選択肢・理由・観察・考察を**項目ごとに**記入してください。これが採点対象です。

## 目次
1. モデルの読み込み
2. 手書き推論 UI（完成コード — 編集不要）
3. 推論結果の記録
4. 考察

---

## この回で学ぶこと

### 「学習データと推論データの分布ズレ」という根本的な問題

第14〜16回では MNIST データセットで学習し，MNIST のテストデータで評価してきた。しかし今回は「自分の手書き文字」という，学習データとは**異なる分布**のデータで推論する。

この問題を **Distribution Shift（分布シフト）** または **Domain Shift** と呼ぶ：

```
【学習時のデータ分布】         【推論時のデータ分布】
MNIST（白背景・黒文字・          自分の手書き（UI キャンバス：
  標準的なサイズ・               黒背景・白文字・
  中央揃え）                     個人差がある文字スタイル）
```

これが AI の実運用での最大の課題の一つだ。

### 前処理の一致がなぜ重要か

モデルは学習時の前処理を「前提」として学習している。推論時に異なる前処理をすると，モデルが意図しない入力を受け取ることになる：

```
学習時: PIL画像 → ToTensor() → [0,1]正規化 → 白地に黒文字
推論時（今回）: Canvas画像（黒地に白文字）→ 色反転 → リサイズ(28,28) → 同じ正規化
```

今回のコードで `arr = 1.0 - arr` （色反転）を行っているのは，この分布ズレを補正するためだ。

### 信頼度（Confidence）の意味

`torch.softmax(logits, dim=1)` の出力は各クラスの「確率」だ。最大値が「信頼度」として表示される。ただし：

- 信頼度が高い ≠ 必ず正解（モデルが確信を持って間違えることもある）
- これを「過信（Overconfidence）」と呼ぶ。深層学習モデルによく見られる問題だ
- 対策：Temperature Scaling, MC Dropout などの不確かさ推定手法がある

### 推論の高速化と本番デプロイ

今回の推論は1枚ずつ行うが，実際のサービスでは：
- バッチ推論（複数画像を一度に処理）
- `model.eval()` + `torch.no_grad()` は必須（速度・メモリ効率）
- ONNX Export（モデルを PyTorch 以外の環境でも動かす標準形式）
- TorchScript（Python 依存をなくして高速化）

卒業研究でも「作ったモデルを実際に動かす」ところまで取り組むと，研究の完成度が大きく上がる。

In [ ]:
%pip install -q torch torchvision

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

DATA_ROOT = "./data"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.ToTensor()
test_dataset = datasets.MNIST(root=DATA_ROOT, train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

## 問題1　モデルの読み込みを読み解く　【説明】
***

### なぜモデルを再読み込みして確認するのか

「保存したモデルを正しく読み込めているか」を確認することは，実際の運用でも重要なステップだ。保存と読み込みに成功していれば，テスト精度が第16回と全く同じ値になるはずだ。

### `load_state_dict` の使い方

```python
model = SimpleCNN().to(device)  # モデルの「構造」を定義（空箱を作る）
model.load_state_dict(          # 学習済みの「重み」を読み込む（中身を詰める）
    torch.load("best_mnist_model.pth", map_location=device)
)
model.eval()  # 推論モードに切り替え（必須！）
```

**`map_location=device` の意味**：GPU で保存したモデルを CPU 環境で読み込む場合（または逆）に，適切なデバイスに重みをマッピングする。これがないと GPU で保存したモデルが CPU 環境で読み込めないエラーが発生する。

### モデル構造の一致が必須

`SimpleCNN` クラスの定義が第16回と**完全に一致している必要がある**。層の数・サイズ・名前が一つでも違うと `load_state_dict` が失敗する。これは「モデルのバージョン管理」が重要な理由だ。

### 課題

下のコードセルには、第16回と同じ `SimpleCNN` クラスの定義と、`best_mnist_model.pth` を読み込んでテスト精度を再確認するコードが **完成形で用意**されています（このセルはローカル Jupyter でも実行できます）。

**各行の `# 説明:` の右に、その行が何をしているかを自分の言葉で書いて**ください（コードは変更しない）。書き終えたら実行し、**テスト正解率が第16回と同じ値**になることを確認してください。

「保存済みモデルを使う」流れは **構造の再定義 → 重みの読み込み → 推論モード** の3段階です。次の問いを意識して説明してください：

- `model = SimpleCNN().to(device)` は何を作っているのか？（なぜ重みを読み込む前に「構造」が必要なのか）
- `load_state_dict(torch.load(...))` は何をしているのか？ `SimpleCNN` の構造が第16回と1つでも違うとなぜ失敗するのか？
- `map_location=device` は何のためにあるのか？
- なぜ推論前に `model.eval()` と `torch.no_grad()` が必要なのか？

In [ ]:
# 完成形です。各行の「# 説明:」に自分の言葉で意味を書いてください。
# （コードは変更しない。説明は AI に書かせず自分で書くこと）

class SimpleCNN(nn.Module):
    """第16回と同一構造でなければ load_state_dict が失敗する。"""
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),   # 説明:（28x28 -> ?）
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 説明:（28 -> 14）
            nn.Conv2d(16, 32, kernel_size=3, padding=1),  # 説明:
            nn.ReLU(),
            nn.MaxPool2d(2),                              # 説明:（14 -> 7）
        )
        self.fc = nn.Sequential(
            nn.Linear(32 * 7 * 7, 128),                   # 説明:（なぜ 32*7*7 ?）
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)                         # 説明:（flatten の意味）
        return self.fc(x)


# === 保存済みモデルの読み込み（完成形）。各行の「# 説明:」を記入 ===
model = SimpleCNN().to(device)                            # 説明:（構造＝空箱を作る）
model.load_state_dict(                                    # 説明:（学習済みの重みを詰める）
    torch.load("best_mnist_model.pth", map_location=device)  # 説明:（map_location の役割）
)
model.eval()                                              # 説明:（なぜ推論前に eval？）

# テスト正解率の再確認（第16回と一致するはず）
correct = total = 0
with torch.no_grad():                                     # 説明:（no_grad の効果）
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
print(f"test accuracy = {correct / total:.4f}")


In [ ]:
# === ✍️ 問題1 解答（採点対象）===
# 主な提出物は上のコードセルへの # 説明: 記入。以下も記入すること。

# (1-a) `SimpleCNN().to(device)` で「構造（空箱）」を先に作る必要があるのはなぜか
answer_1_a = """
"""

# (1-b) `load_state_dict` が、構造が第16回と1つでも違うと失敗するのはなぜか
answer_1_b = """
"""

# (1-c) `map_location=device` の役割
answer_1_c = """
"""

# (1-d) 推論前に `model.eval()` と `torch.no_grad()` が必要な理由
answer_1_d = """
"""

# (1-e) 読み込んだモデルのテスト正解率（第16回と一致したか）
answer_1_e = """
"""



## 手書き推論 UI
***
以下のセルは **完成コード** です。編集せず実行してください。

Canvas に 0〜9 の数字を書き，「推論する」ボタンを押すと予測結果が表示されます。

> **Canvas は黒背景・白線** です。MNIST は白背景・黒線のため，推論時に `1.0 - arr` で**色を反転**しています。これが Distribution Shift への対処法の一つだ。

### 推論の処理フロー（内部で何が起きているか）

```
1. Canvas からピクセルデータを取得（PNG → base64 → PIL Image）
2. グレースケールに変換（カラー情報は不要）
3. 28×28 にリサイズ（MNIST と同じサイズに）
4. 値を [0,1] に正規化（255で割る）
5. 色を反転（1.0 - arr）← Distribution Shift の補正
6. Tensor に変換して shape を (1, 1, 28, 28) に
7. model(tensor) で推論 → logits (1, 10)
8. softmax で確率に変換 → argmax で予測クラスを取得
```

In [ ]:
# 手書き推論UI（完成コード — 編集不要）
import numpy as np
from IPython.display import display, HTML
import base64
from PIL import Image
import io
from google.colab import output


def predict_digit(img_data_url):
    header, data = img_data_url.split(",", 1)
    img_bytes = base64.b64decode(data)
    img = Image.open(io.BytesIO(img_bytes)).convert("L")

    img = img.resize((28, 28), Image.LANCZOS)
    arr = np.array(img).astype("float32") / 255.0
    arr = 1.0 - arr  # Canvas=黒地白線 → MNIST=白地黒線 に反転

    tensor = torch.tensor(arr).unsqueeze(0).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1)
        digit = int(probs.argmax(dim=1).item())
        confidence = float(probs.max().item()) * 100
    return digit, confidence


def _on_predict(data_url):
    digit, conf = predict_digit(data_url)
    print(f"予測: {digit}  信頼度: {conf:.1f}%")


output.register_callback("predict_from_js", _on_predict)

html_code = """
<style>
  body { font-family: sans-serif; }
  #canvas {
    border: 3px solid #333;
    border-radius: 8px;
    cursor: crosshair;
    background: black;
    display: block;
    margin: 10px 0;
  }
  .btn {
    padding: 10px 24px;
    margin: 4px;
    font-size: 16px;
    border: none;
    border-radius: 6px;
    cursor: pointer;
  }
  #predictBtn { background: #4CAF50; color: white; }
  #clearBtn   { background: #f44336; color: white; }
  #result {
    font-size: 32px;
    font-weight: bold;
    margin-top: 12px;
    min-height: 40px;
    color: #1a73e8;
  }
  #confidence { font-size: 16px; color: #555; }
</style>

<h3>数字を書いてください（0〜9）</h3>
<canvas id="canvas" width="280" height="280"></canvas>

<div>
  <button class="btn" id="predictBtn" onclick="predict()">推論する</button>
  <button class="btn" id="clearBtn"   onclick="clearCanvas()">クリア</button>
</div>

<div id="result">ここに結果が表示されます</div>
<div id="confidence"></div>

<script>
  const canvas = document.getElementById("canvas");
  const ctx    = canvas.getContext("2d");

  ctx.fillStyle = "black";
  ctx.fillRect(0, 0, 280, 280);
  ctx.strokeStyle = "white";
  ctx.lineWidth   = 20;
  ctx.lineCap     = "round";
  ctx.lineJoin    = "round";

  let drawing = false;
  let lastX = 0, lastY = 0;

  function getPos(e) {
    const rect = canvas.getBoundingClientRect();
    if (e.touches) {
      return {
        x: e.touches[0].clientX - rect.left,
        y: e.touches[0].clientY - rect.top
      };
    }
    return { x: e.clientX - rect.left, y: e.clientY - rect.top };
  }

  canvas.addEventListener("mousedown",  e => { drawing = true; const p = getPos(e); lastX = p.x; lastY = p.y; });
  canvas.addEventListener("mousemove",  e => {
    if (!drawing) return;
    const p = getPos(e);
    ctx.beginPath();
    ctx.moveTo(lastX, lastY);
    ctx.lineTo(p.x, p.y);
    ctx.stroke();
    lastX = p.x; lastY = p.y;
  });
  canvas.addEventListener("mouseup",   () => drawing = false);
  canvas.addEventListener("mouseleave",() => drawing = false);

  canvas.addEventListener("touchstart",  e => { e.preventDefault(); drawing = true; const p = getPos(e); lastX = p.x; lastY = p.y; });
  canvas.addEventListener("touchmove",   e => { e.preventDefault(); if (!drawing) return; const p = getPos(e); ctx.beginPath(); ctx.moveTo(lastX, lastY); ctx.lineTo(p.x, p.y); ctx.stroke(); lastX = p.x; lastY = p.y; });
  canvas.addEventListener("touchend",    e => { e.preventDefault(); drawing = false; });

  function clearCanvas() {
    ctx.fillStyle = "black";
    ctx.fillRect(0, 0, 280, 280);
    document.getElementById("result").innerText = "ここに結果が表示されます";
    document.getElementById("confidence").innerText = "";
  }

  function predict() {
    const dataURL = canvas.toDataURL("image/png");
    google.colab.kernel.invokeFunction("predict_from_js", [dataURL], {});
  }
</script>
"""

display(HTML(html_code))
print("キャンバスを表示しました。数字を書いて「推論する」を押してください。")


## 問題2　前処理の理解と推論結果の記録　【骨格+選択】
***

### 記録すべき情報

UI で0〜9の数字を書いて推論し，以下の情報を記録してください：
- **正解ラベル**: 書いた数字
- **予測**: モデルが予測した数字
- **信頼度（%）**: モデルの確信度

### 誤認識パターンの分析

誤認識が起きた場合，以下を確認しよう：
1. **信頼度は高いか低いか**: 低信頼度（60%以下）の誤認識は「曖昧な入力」が原因
2. **どの数字に間違えたか**: 第16回の混同行列と一致するパターンか
3. **書き方は普通か**: 独特な書き方（例：欧風の「1」に横棒を付けるなど）が原因なことも

### Distribution Shift の観察

自分の手書き文字の精度が，第16回のテストデータ（MNIST）の精度より低い場合，それは Distribution Shift の影響だ。なぜ低くなるかを考えてみよう：
- 文字の太さの違い
- 文字の傾き
- キャンバス内での位置（MNIST は中央揃え）

### 課題

下のコードセルには2つのものが **完成形**で用意されています：

1. **【骨格】前処理関数 `preprocess_for_mnist`** … UI 内部と同じ処理（グレースケール化 → 28×28 リサイズ → 正規化 → 色反転）を関数にまとめたものです。なお、Distribution Shift 補正の核心である **色反転の1行はあなたが書きます**（`# ★あなたが書く★`）。加えて `★` の **二値化しきい値（`BINARIZE` / `THRESHOLD`）** を変えて、線が薄い/濃いときに二値化が効くか試せます。
2. **推論結果の記録テーブル** … `records` に結果を手入力すると DataFrame と正解率が出ます。

**手順（Colab 推奨）**: 上の UI で **0〜9 を各1回以上**書いて推論し、表示された `(予測, 信頼度)` と自分が書いた数字（正解）を `records` に記入してください。ローカルで UI が使えない場合は、誤認識が起きた状況を想像して考察に進んでください。

> **設計判断（誤認識の原因を選ぶ）**: 手書き数字が誤認識されたとき、考えられる原因を次の **6つから1つ以上選び**、その **対処法**を解答用コードセルに書いてください。**当てはまらない選択肢も混ざっています**。
>
> - **(A) 前処理のずれ** … 中心化やスケール（大きさ・位置）が MNIST と合っていない
> - **(B) 学習不足** … モデルの学習エポックや表現力が足りない
> - **(C) 分布外（Distribution Shift）** … 訓練データ（MNIST）と書き方の癖が違う
> - **(D) 線が細すぎ／太すぎ** … 画線の太さが MNIST と大きく異なる
> - **(E) 色反転の漏れ／誤り** … Canvas（黒地白線）を MNIST（白地黒線）に反転できていない
> - **(F) モデルの過信（Overconfidence）** … 信頼度は高いのに誤る。softmax 確率が校正されていない
>
> ヒント：信頼度が高いのに間違える／低くて間違える、どちらだったかも手がかりになります。

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image

# === 【骨格】推論用の前処理（完成形）。UI 内部と同じ処理を関数化したものです ===
# 変更してよいのは ★ の BINARIZE / THRESHOLD だけです。
def preprocess_for_mnist(pil_img, binarize=False, threshold=0.5):
    """PIL 画像を MNIST 形式 (1, 1, 28, 28) のテンソルに変換する。"""
    img = pil_img.convert("L")                       # グレースケール化（色情報は不要）
    img = img.resize((28, 28), Image.LANCZOS)        # 28x28 にリサイズ（MNIST と同じ）
    arr = np.array(img).astype("float32") / 255.0    # [0,1] に正規化
    # ★あなたが書く★：色反転（Distribution Shift 補正）。Canvas=黒地白線 → MNIST=白地黒線
    #   ヒント: 値を 0↔1 で反転する。arr = 1.0 - arr
    arr = ___
    # === ★ここを変えて実験する★：二値化のしきい値（線が薄い/濃いとき有効か試す） ===
    if binarize:
        arr = (arr > threshold).astype("float32")
    tensor = torch.tensor(arr).unsqueeze(0).unsqueeze(0).to(device)
    return tensor


# === 推論結果の記録（UI で書いた数字の結果をここに手入力） ===
# Colab の UI で 0〜9 を書いて推論し、(正解 label, 予測 pred, 信頼度 confidence) を記入します。
records = [
    # {"label": 0, "pred": 0, "confidence": 98.5},
    # {"label": 1, "pred": 7, "confidence": 55.2},  # ← 誤認識の例
    # ... 0〜9 を各1回以上記入してください ...
]

if records:
    df = pd.DataFrame(records)
    acc = (df["label"] == df["pred"]).mean()
    print(df.to_string(index=False))
    print(f"\n自作数字の正解率 = {acc:.2%}")
else:
    print("records に UI での推論結果（label, pred, confidence）を記入してください（Colab 推奨）。")


In [ ]:
# === ✍️ 問題2 解答（採点対象）===
import pandas as pd


# (2-a) 推論結果（Colab の UI で 0〜9 を書いた結果）
# メインの記入先: 上のコードセルの records。以下はメモ列などの補足用（任意）
inference_log = pd.DataFrame([
    {'label': 0, 'pred': None, 'confidence_pct': None, 'correct': None, 'memo': None},
    {'label': 1, 'pred': None, 'confidence_pct': None, 'correct': None, 'memo': None},
])

# (2-b) 自作数字の正解率
answer_2_b = """
"""

# (2-c) 二値化（`BINARIZE` / `THRESHOLD`）を試した場合の変化（任意）
answer_2_c = """
"""

# (2-d) 設計判断：誤認識の原因として選んだもの（A〜F、複数可）：(　)
# (A) 前処理のずれ
# (B) 学習不足
# (C) 分布外（Distribution Shift）
# (D) 線が細すぎ／太すぎ
# (E) 色反転の漏れ／誤り
# (F) モデルの過信
design2_choice = ""

# (2-e) その対処法
answer_2_e = """
"""



## 問題3　総合考察　【考察】
***

### この問いに答えることの意味

単に「コードを動かした」だけでなく，**なぜそうなったか** を自分の言葉で説明できることが，真の理解の証明だ。卒業研究では実験結果の考察は必須であり，「モデルが良い/悪かった → なぜか → どうすれば改善できるか」という思考の流れを練習しよう。

### 考察のための視点

**① CNN が MLP より精度が高い理由について**：
- 畳み込み層は「局所的な特徴」を検出する。数字の「丸み」「直線」「交差点」など，28×28 の中の小さなパターンを見つけられる
- 重みを画像全体で共有（重み共有）するため，パラメータが少ない割に表現力が高い
- MLP は784個のピクセルをバラバラに扱うため，「隣接するピクセルの関係性」を学習しにくい

**② 自作数字の誤認識原因について**：
- MNIST は特定のスタイルの手書き文字だけで学習している
- 自分の書き方がトレーニングデータの分布から外れると精度が下がる
- 「前処理（リサイズ，色反転，正規化）が正しく行われているか」も確認が必要

**③ 精度向上のアイデアについて（選択肢）**：
- **Data Augmentation（データ拡張）**: 学習データを回転・拡大・ノイズ追加などで増やす → テストへの汎化性が上がる
- **Dropout**: 学習時に一部のニューロンをランダムに無効化 → 過学習を防ぐ
- **Batch Normalization**: 各層の出力を正規化 → 学習が安定・高速化
- **エポック数を増やす**: 損失曲線を見て，まだ収束していないなら有効
- **より深い CNN**: VGG, ResNet など実績のあるアーキテクチャを使う

### 課題

この問題は **考察中心**です。まず下のコードセル（**完成形・ローカルでも実行可**）を実行すると、読み込んだモデルで MNIST テスト画像を数枚推論し、正解・予測・信頼度が表示されます。これを「観察の材料」にしてください。

そのうえで、下の **✍️ 解答用コードセル** に、次の3点について自分の言葉で考察を書いてください（各2〜4文程度）：

1. **第16回で CNN が MLP（LinearNet, DeepMLP）より精度が高かった理由**を、CNN の仕組み（局所特徴の抽出・重み共有）を踏まえて説明してください。
2. **誤認識が起きる（起きた）原因**を、**Distribution Shift（分布シフト）**や**前処理（リサイズ・色反転・正規化）**の観点から考察してください。問題2で選んだ原因とも関連づけてください。
3. **精度をさらに上げるために試せる手法を2つ**挙げ、それぞれ「なぜ効果があると思うか」を説明してください（例：Data Augmentation / Dropout / Batch Normalization / エポック増 / より深い CNN）。

> **ポイント**: 「正解率が高かった/低かった」という事実だけでなく、「**なぜそうなったか**」「**どう改善できるか**」の考察が重要です。これが理解の証跡になります。


In [ ]:
# === 完成形（ローカルでも実行可）：MNIST テスト画像で推論を観察し、考察の材料にする ===
# 読み込んだ model（問題1）を使って数枚を推論し、正解・予測・信頼度を表示します。
model.eval()
images, labels = next(iter(test_loader))
images = images.to(device)
with torch.no_grad():
    probs = torch.softmax(model(images), dim=1)
    preds = probs.argmax(dim=1)
    confs = probs.max(dim=1).values

print("MNIST テスト画像の推論結果（先頭8枚）:")
for i in range(8):
    mark = "OK" if preds[i].item() == labels[i].item() else "NG"
    print(f"[{mark}] 正解={labels[i].item()}  予測={preds[i].item()}  信頼度={confs[i].item() * 100:5.1f}%")

# 観察の補足：高信頼度で間違える例があるか、信頼度の分布はどうかを見て、下の解答用コードセルの考察に使ってください。


In [ ]:
# === ✍️ 問題3 解答（採点対象）===

# (3-a) 観察メモ（上のセルの出力／自作数字の傾向。OK・NG や信頼度について気づいたこと）
observation_3_a = """
"""

# (3-b) 考察1（CNN が MLP より精度が高かった理由。局所特徴・重み共有に触れて）
reflection1 = """
"""

# (3-c) 考察2（誤認識の原因。Distribution Shift・前処理の観点で。問題2で選んだ原因と関連づけて）
reflection2 = """
"""

# (3-d) 考察3（精度を上げる手法2つと、それぞれなぜ効くか）
reflection3 = """
"""
# 手法1
improvement_1 = ""
# 手法2
improvement_2 = ""

